#Tugas 2 lanjutan dari tugas pertama "kata unik"

In [1]:
!pip install pandas openpyxl numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 34.9 MB/s  0:00:00 168.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 35.4 MB/s  0:00:00s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn]0m 5/6 [scikit-learn]]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

df.head()

,id,isi_berita,label
0,1,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",sport
1,2,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",sport
2,3,Di tengah viral insiden cepirit pada ajang Hyr...,sport
3,4,Dejan Fedinansyah/Felisha Alberta Nathaniel Pa...,sport
4,5,Bagas Maulana/Apriyani Rahayu tak minder meski...,sport


In [4]:
print("Jumlah data :", len(df))

print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom:
Index(['id', 'isi_berita', 'label'], dtype='str')

Jumlah data per label:
label
sport      100
finance    100
Name: count, dtype: int64


In [6]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [7]:
df["jumlah_kata_asli"] = df["isi_berita"].astype(str).apply(
    lambda x: len(x.split())
)

df[["id", "jumlah_kata_asli"]].head()

,id,jumlah_kata_asli
0,1,320
1,2,323
2,3,256
3,4,238
4,5,402


In [8]:
total_kata = df["jumlah_kata_asli"].sum()

print("Total seluruh kata :", total_kata)
print(df["jumlah_kata_asli"].describe())

Total seluruh kata : 66951
count    200.000000
mean     334.755000
std      161.288234
min       42.000000
25%      237.750000
50%      301.500000
75%      392.250000
max      963.000000
Name: jumlah_kata_asli, dtype: float64


In [10]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

In [11]:
kamus_asing = {
    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

In [12]:
def preprocessing(text):
    # Pastikan berupa string
    text = str(text)

    # =========================
    # CASE FOLDING
    # =========================
    text = text.lower()

    # =========================
    # HAPUS URL
    # =========================
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )

    # =========================
    # HAPUS EMAIL
    # =========================
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )

    # =========================
    # HAPUS ANGKA
    # =========================
    text = re.sub(
        r'\d+',
        ' ',
        text
    )

    # =========================
    # HAPUS TANDA BACA,
    # SIMBOL DAN EMOTICON
    # =========================
    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )

    # =========================
    # HAPUS SPASI BERLEBIH
    # =========================
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # =========================
    # TOKENISASI
    # =========================
    kata = text.split()

    hasil = []

    for token in kata:

        # Membakukan kata tidak baku
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]

        # Bahasa asing → Indonesia
        if token in kamus_asing:
            token = kamus_asing[token]

        hasil.append(token)

    return " ".join(hasil)

In [13]:
df["berita_clean"] = df["isi_berita"].apply(
    preprocessing
)

df[
    [
        "id",
        "isi_berita",
        "berita_clean"
    ]
].head()

,id,isi_berita,berita_clean
0,1,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",dalam dua seri terakhir motogp marc marquez su...
1,2,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",alwi farhan moh zaki ubaidillah dan muhamad yu...
2,3,Di tengah viral insiden cepirit pada ajang Hyr...,di tengah viral insiden cepirit pada ajang hyr...
3,4,Dejan Fedinansyah/Felisha Alberta Nathaniel Pa...,dejan fedinansyah felisha alberta nathaniel pa...
4,5,Bagas Maulana/Apriyani Rahayu tak minder meski...,bagas maulana apriyani rahayu tak minder meski...


In [14]:
df["jumlah_kata_clean"] = df["berita_clean"].apply(
    lambda x: len(x.split())
)

In [15]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 66951
Total kata setelah preprocessing : 65311


In [16]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 66951
Total kata setelah preprocessing : 65311


In [19]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print("Jumlah seluruh kata:", len(semua_kata))
print("Jumlah kata unik:", len(kata_unik))

Jumlah seluruh kata: 65311
Jumlah kata unik: 7276


In [20]:
kata_unik[:100]

['a',
 'aaa',
 'aau',
 'ab',
 'abad',
 'abadi',
 'abal',
 'abdullah',
 'abdurrahkman',
 'abha',
 'abraham',
 'abrahham',
 'absen',
 'absorber',
 'abu',
 'academy',
 'acara',
 'acaranya',
 'accelerating',
 'acceptance',
 'access',
 'accessories',
 'accident',
 'accreditation',
 'acd',
 'ace',
 'aceh',
 'achadie',
 'achilles',
 'achmad',
 'acosta',
 'acuan',
 'ada',
 'adalah',
 'adam',
 'adanya',
 'adaptasi',
 'adaptif',
 'adapun',
 'adb',
 'adcp',
 'ade',
 'adelaide',
 'adelguer',
 'adella',
 'adhang',
 'adhi',
 'adhipramana',
 'adhitama',
 'adi',
 'adibha',
 'adik',
 'adil',
 'adinata',
 'aditya',
 'administrasi',
 'administratif',
 'administrator',
 'adp',
 'adr',
 'adrenalin',
 'adrian',
 'adrianto',
 'adshila',
 'aduh',
 'adventure',
 'aerox',
 'afc',
 'aff',
 'afiliasi',
 'afirmasi',
 'agak',
 'agar',
 'agenda',
 'agius',
 'agn',
 'agraria',
 'agreement',
 'agresif',
 'agribisnis',
 'agung',
 'agus',
 'agusman',
 'agustus',
 'ahhn',
 'ahi',
 'ahli',
 'ahmad',
 'ahren',
 'ahy',
 'ah

In [21]:
print("Jumlah kata unik:", len(kata_unik))

Jumlah kata unik: 7276


In [22]:
print("Jumlah kata unik:", len(kata_unik))

Jumlah kata unik: 7276


In [24]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["berita_clean"],
    df["label_num"],
    test_size=40,
    random_state=42,
    stratify=df["label_num"]
)

print("Jumlah training:", len(X_train_text))
print("Jumlah testing:", len(X_test_text))

Jumlah training: 160
Jumlah testing: 40


In [25]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


In [27]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 3174


In [28]:
print("Ukuran X_train_tfidf:", X_train_tfidf.shape)
print("Ukuran X_test_tfidf :", X_test_tfidf.shape)

Ukuran X_train_tfidf: (160, 3174)
Ukuran X_test_tfidf : (40, 3174)


In [29]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df["label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.head()

,aaa,abadi,abraham,absen,acara,acd,achilles,acosta,ada,adalah,...,yogyakarta,youtube,yudha,yudhi,yudo,yusuf,zaki,zarco,zein,zona
0,0.0,0.0,0.0,0.0,0.0,0.0583,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.158855,0.048148,0.052697,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000000,0.020662,0.045229,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
print(
    "Ukuran DataFrame TF-IDF:",
    tfidf_train_df.shape
)

Ukuran DataFrame TF-IDF: (160, 3174)


In [31]:
tfidf_train_df["label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

print("Data TF-IDF berhasil disimpan.")

Data TF-IDF berhasil disimpan.


In [32]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

for kelas in sorted(
    y_train.unique()
):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]

    # Berapa dokumen kelas tersebut
    # mengandung sebuah kata
    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )

    # Jumlah dokumen dalam kelas
    jumlah_dokumen_class = (
        X_class.shape[0]
    )

    # Kepadatan kata pada kelas
    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )

    class_space_density += (
        class_density
    )

In [33]:
epsilon = 1e-12

icsdf = np.log(
    (jumlah_kelas + epsilon)
    /
    (class_space_density + epsilon)
)

df_icsdf = pd.DataFrame({
    "kata": features,
    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
6,achilles,4.382027
1898,merenung,4.382027
1900,merespons,4.382027
1901,merk,4.382027
1877,menyenangkan,4.382027
1878,menyentuh,4.382027
1882,menyimpan,4.382027
1886,menyumbang,4.382027
1863,menyaksikan,4.382027
1867,menyapu,4.382027


In [34]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

print(
    "Ukuran training:",
    X_train_icsdf.shape
)

print(
    "Ukuran testing:",
    X_test_icsdf.shape
)

Ukuran training: (160, 3174)
Ukuran testing: (40, 3174)


In [35]:
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

TOP_K = 100

top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

kata_penting = features[
    top_index
]

print(
    "Jumlah kata penting:",
    len(kata_penting)
)

print(
    kata_penting[:30]
)

Jumlah kata penting: 100
['marquez' 'purbaya' 'emas' 'pertandingan' 'lrt' 'marc' 'motogp' 'poin'
 'asian' 'pramono' 'beras' 'bezzecchi' 'martin' 'djarum' 'kapal' 'aku'
 'atlet' 'fortifikasi' 'balapan' 'peserta' 'artikel' 'stasiun' 'suahasil'
 'medali' 'keuangan' 'fiskal' 'rp' 'nathan' 'pejabat' 'manggarai']


In [36]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 3174)
Sesudah ICSDF: (160, 100)


In [37]:
pca = PCA(
    n_components=20,
    random_state=42
)

In [38]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

print(
    "Ukuran PCA training:",
    X_train_pca.shape
)

print(
    "Ukuran PCA testing:",
    X_test_pca.shape
)

Ukuran PCA training: (160, 20)
Ukuran PCA testing: (40, 20)


In [39]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

print(nama_pc)

['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20']


In [40]:
df_train_reduksi = pd.DataFrame(
    X_train_pca,
    columns=nama_pc
)

df_train_reduksi["label"] = (
    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.023739,0.101394,-0.049422,-0.063356,0.086383,0.007532,0.083068,0.233131,-0.015137,-0.009478,...,0.083307,-0.019954,0.010011,-0.020740,-0.049747,-0.027655,-0.029153,-0.039628,-0.005805,0
1,-0.385790,-0.327441,0.333557,0.059923,-0.010368,0.037999,0.041251,-0.088813,-0.002219,0.028659,...,-0.002432,0.015914,-0.258360,0.008273,-0.017450,0.036909,0.000441,-0.031174,-0.020611,1
2,0.044241,0.094802,-0.194353,-0.355199,-0.077077,0.467351,0.235653,0.258455,-0.147006,-0.263681,...,0.770157,-0.242350,-0.192999,0.205340,0.080552,0.014194,-0.047052,0.126539,-0.051390,0
3,-0.442231,-0.380277,0.393622,0.077757,-0.005886,0.049104,0.037874,-0.090491,0.004265,0.034841,...,0.002173,0.028855,-0.256707,0.006912,-0.012023,0.006613,-0.001652,-0.009177,-0.006460,1
4,-0.602909,-0.536810,0.569808,0.123258,-0.015111,0.184759,-0.022220,0.011972,-0.215266,-0.086353,...,0.067581,-0.055473,0.742603,0.187519,0.150997,-0.025623,-0.005352,0.058738,0.049230,1


In [41]:
df_test_reduksi = pd.DataFrame(
    X_test_pca,
    columns=nama_pc
)

df_test_reduksi["label"] = (
    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.093528,-0.008930,-0.026392,-0.014438,0.007135,-0.055266,0.005843,0.043934,-0.004545,-0.007255,...,-0.031915,0.008970,0.013543,-0.052270,-0.053962,0.007206,-0.000616,-0.011047,-0.008313,0
1,-0.690028,-0.625900,0.696743,0.190475,-0.014567,0.239035,-0.027859,-0.090097,-0.048399,0.027761,...,0.022128,-0.018620,-0.396658,0.086031,0.056908,0.212722,0.033930,-0.143442,-0.053950,1
2,-0.147954,-0.082530,0.061283,-0.020260,-0.006656,-0.056844,-0.048688,0.048778,-0.058337,0.043137,...,-0.026939,-0.021080,0.201045,-0.161813,-0.265954,-0.315363,-0.184145,0.551493,0.424346,1
3,-0.019334,0.064004,-0.158421,-0.196330,-0.090484,-0.040796,-0.041731,-0.048561,-0.006393,-0.159874,...,-0.088879,0.252501,-0.026124,-0.116490,0.087563,-0.000726,-0.006694,-0.011519,-0.022514,1
4,-0.008530,0.041859,-0.065064,-0.028745,0.018857,-0.055054,0.015177,0.070613,-0.004682,-0.005006,...,-0.034938,0.001964,0.017133,-0.067287,-0.082270,0.016842,-0.012633,-0.020134,-0.018853,0


In [42]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

print("Data reduksi berhasil disimpan.")

Data reduksi berhasil disimpan.


In [43]:
print(
    "Total training:",
    len(df_train_reduksi)
)

print(
    "Total testing:",
    len(df_test_reduksi)
)

print(
    "Total keseluruhan:",
    len(df_train_reduksi)
    + len(df_test_reduksi)
)

Total training: 160
Total testing: 40
Total keseluruhan: 200


In [44]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

print("Data reduksi berhasil disimpan.")

Data reduksi berhasil disimpan.


In [45]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


In [47]:
print("Kolom training:")
print(df_train_reduksi.columns.tolist())

print("\nKolom testing:")
print(df_test_reduksi.columns.tolist())

Kolom training:
['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20', 'label']

Kolom testing:
['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20', 'label']


In [46]:
print(
    "Total training:",
    len(df_train_reduksi)
)

print(
    "Total testing:",
    len(df_test_reduksi)
)

print(
    "Total data:",
    len(df_train_reduksi)
    + len(df_test_reduksi)
)

Total training: 160
Total testing: 40
Total data: 200
